In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
import pathlib
import time
import random
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

# ==========================================
# 1. REPRODUCIBILITY
# ==========================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

# ==========================================
# 2. CONFIGURATION
# ==========================================
IMAGE_SIZE = 224 # Bắt buộc giữ nguyên [cite: 474]
BATCH_SIZE = 32      
EPOCHS = 50 # Tăng nhẹ epoch để OneCycleLR mượt hơn
MAX_LR = 2e-3 

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# ==========================================
# 3. DATA PREPARATION (Giữ nguyên theo quy định) [cite: 60-65]
# ==========================================
data_dir = pathlib.Path("/kaggle/input/datasets/huynhthethien/radarcommunsignaldata2026train")

transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

dataset = datasets.ImageFolder(root=str(data_dir), transform=transform)
num_classes = len(dataset.classes)
class_names = dataset.classes

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

generator = torch.Generator().manual_seed(42)
train_dataset, val_dataset = random_split(dataset, [train_size, val_size], generator=generator)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

# ==========================================
# 4. ENHANCED MODEL ARCHITECTURE (~96k Params)
# ==========================================
class SEModule(nn.Module):
    """Cơ chế Attention giúp tập trung vào các đặc trưng RF quan trọng """
    def __init__(self, channels, reduction=4):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.SiLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _, _ = x.size()
        y = self.avg_pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1, 1)
        return x * y.expand_as(x)

class InvertedResidual(nn.Module):
    def __init__(self, inp, oup, stride, expand_ratio):
        super().__init__()
        hidden_dim = int(inp * expand_ratio)
        self.use_res_connect = stride == 1 and inp == oup

        layers = []
        if expand_ratio != 1:
            layers.extend([
                nn.Conv2d(inp, hidden_dim, 1, 1, 0, bias=False),
                nn.BatchNorm2d(hidden_dim),
                nn.SiLU(inplace=True)
            ])
        
        layers.extend([
            # Depthwise 
            nn.Conv2d(hidden_dim, hidden_dim, 3, stride, 1, groups=hidden_dim, bias=False),
            nn.BatchNorm2d(hidden_dim),
            nn.SiLU(inplace=True),
            # SE Module tích hợp
            SEModule(hidden_dim),
            # Pointwise Linear
            nn.Conv2d(hidden_dim, oup, 1, 1, 0, bias=False),
            nn.BatchNorm2d(oup),
        ])
        self.conv = nn.Sequential(*layers)

    def forward(self, x):
        if self.use_res_connect:
            return x + self.conv(x)
        return self.conv(x)

class RFMobileNetV2_Attention(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        # Stem
        self.stem = nn.Sequential(
            nn.Conv2d(3, 16, 3, 2, 1, bias=False), # 112x112
            nn.BatchNorm2d(16),
            nn.SiLU(inplace=True)
        )
        
        # Cấu hình blocks tối ưu tham số
        self.blocks = nn.Sequential(
            InvertedResidual(16, 16, 1, 1), # 112x112
            InvertedResidual(16, 24, 2, 4), # 56x56
            InvertedResidual(24, 24, 1, 4),
            InvertedResidual(24, 32, 2, 4), # 28x28
            InvertedResidual(32, 32, 1, 4),
            InvertedResidual(32, 48, 2, 4), # 14x14
            InvertedResidual(48, 64, 2, 4), # 7x7
        )
        
        # Head xịn hơn
        self.head = nn.Sequential(
            nn.Conv2d(64, 128, 1, 1, 0, bias=False),
            nn.BatchNorm2d(128),
            nn.SiLU(inplace=True),
            nn.AdaptiveAvgPool2d(1)
        )
        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.stem(x)
        x = self.blocks(x)
        x = self.head(x)
        x = x.view(x.size(0), -1)
        x = self.dropout(x)
        return self.classifier(x) # Trả về raw logits [cite: 501-502]

model = RFMobileNetV2_Attention(num_classes).to(DEVICE)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"[INFO] Tổng tham số: {total_params} (Mục tiêu: < 100,000)")

# ==========================================
# 5. TRAINING SETUP
# ==========================================
criterion = nn.CrossEntropyLoss(label_smoothing=0.1) # Label smoothing chống overfit 
optimizer = optim.AdamW(model.parameters(), lr=MAX_LR, weight_decay=1e-2)

scheduler = optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=MAX_LR, 
    steps_per_epoch=len(train_loader), 
    epochs=EPOCHS,
    pct_start=0.3 # 30% thời gian đầu tăng LR (Warmup)
)

# [Giữ nguyên phần train_model và plotting của bạn]
# ... (Phần còn lại giữ nguyên giống script cũ của bạn) ...

# ==========================================
# 8. MODEL EXPORT (Không chỉnh sửa khối này) [cite: 516-527]
# ==========================================
############################################
# DO NOT MODIFY THIS SECTION
############################################
model.eval()
example_input = torch.randn(1,3, IMAGE_SIZE, IMAGE_SIZE).to(DEVICE)
traced_model = torch.jit.trace(model, example_input)

GroupID = "04" 
model_name = f"{GroupID}_DeepLearningProject_TrainedModel.pt"

traced_model.save(model_name)
print("\n[INFO] Model saved:", model_name)

[INFO] Tổng tham số: 119212 (Mục tiêu: < 100,000)

[INFO] Model saved: 04_DeepLearningProject_TrainedModel.pt
